In [2]:
import pandas as pd

pd.set_option('display.max_columns', None)

In [3]:
df = pd.read_csv('../data/raw/customer_orders_raw.csv', parse_dates=['OrderDate'])
df.head()

,CustomerID,FirstName,LastName,SalesOrderID,OrderDate,TotalDue,TerritoryName
0,11000,Jon,Yang,43793,2011-06-21,3756.9890,Australia
1,11000,Jon,Yang,51522,2013-06-20,2587.8769,Australia
2,11000,Jon,Yang,57418,2013-10-03,2770.2682,Australia
3,11001,Eugene,Huang,43767,2011-06-17,3729.3640,Australia
4,11001,Eugene,Huang,51493,2013-06-18,2674.0227,Australia


In [4]:
print(df.shape)
df.info()

(31465, 7)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31465 entries, 0 to 31464
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   CustomerID     31465 non-null  int64         
 1   FirstName      31465 non-null  object        
 2   LastName       31465 non-null  object        
 3   SalesOrderID   31465 non-null  int64         
 4   OrderDate      31465 non-null  datetime64[ns]
 5   TotalDue       31465 non-null  float64       
 6   TerritoryName  31465 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(2), object(3)
memory usage: 1.7+ MB


In [5]:
snapshot_date = pd.Timestamp('2014-06-30')

rfm = df.groupby('CustomerID').agg(
    FirstName=('FirstName', 'first'),
    LastName=('LastName', 'first'),
    TerritoryName=('TerritoryName', 'first'),
    LastOrderDate=('OrderDate', 'max'),
    Frequency=('SalesOrderID', 'count'),
    Monetary=('TotalDue', 'sum')
).reset_index()

rfm['Recency'] = (snapshot_date - rfm['LastOrderDate']).dt.days

rfm.head()

,CustomerID,FirstName,LastName,TerritoryName,LastOrderDate,Frequency,Monetary,Recency
0,11000,Jon,Yang,Australia,2013-10-03,3,9115.1341,270
1,11001,Eugene,Huang,Australia,2014-05-12,3,7054.1875,49
2,11002,Ruben,Torres,Australia,2013-07-26,3,8966.0143,339
3,11003,Christy,Zhu,Australia,2013-10-10,3,8993.9155,263
4,11004,Elizabeth,Johnson,Australia,2013-10-01,3,9056.5911,272


In [6]:
print(rfm.shape)
rfm.describe()

(19119, 8)


,CustomerID,LastOrderDate,Frequency,Monetary,Recency
count,19119.000000,19119,19119.000000,19119.000000,19119.000000
mean,20559.000000,2013-12-21 17:34:49.502588928,1.645745,6444.729647,190.267483
min,11000.000000,2011-05-31 00:00:00,1.000000,1.518300,0.000000
25%,15779.500000,2013-10-10 00:00:00,1.000000,60.752900,85.000000
50%,20559.000000,2014-01-16 00:00:00,1.000000,606.622900,165.000000
75%,25338.500000,2014-04-06 00:00:00,2.000000,3119.149400,263.000000
max,30118.000000,2014-06-30 00:00:00,28.000000,989184.082000,1126.000000
std,5519.324234,NaN,1.457054,43756.276004,150.423605


In [7]:
rfm['Churned'] = (rfm['Recency'] > 180).astype(int)

rfm['Churned'].value_counts()

Churned
0    10354
1     8765
Name: count, dtype: int64